# Stack Models

## Set Up

Import packages/libraries

In [ ]:
import sys
import warnings

sys.path.append("../")
from src.data_utils import get_data, get_models
from src.config import SEED, BASE_PATH
from src.nn_model import load_nn_clf
import joblib
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

print(f"Path: {BASE_PATH}")

Import data/models

In [ ]:
# Data
DATA_DICT = get_data(is_nomo=False)


# Models
model_dir = BASE_PATH / "models" / "trained"
## ADD NN LATER
model_prefix_list = ["lgbm", "xgb", "knn", "svc", "nn"]
MODEL_DICT = {}

## Base models
MODEL_DICT = get_models(model_prefix_list, model_dir)

N_SPLITS = 5

## Build Model

In [ ]:
## Train
X_train = DATA_DICT["X_train"]
y_train = DATA_DICT["y_train"]
# Fit stack
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
estimators = list(MODEL_DICT.items())
stack_model = StackingClassifier(
    estimators=estimators,
    cv=skf,
    passthrough=True,
    final_estimator=LogisticRegression(random_state=SEED),
    n_jobs=-1,
)
stack_model.fit(X_train, y_train.values.ravel())

### Export Model ####
model_export_path = model_dir / "stack.joblib"
if model_export_path.exists():
    warnings.warn(f"Over-writing models at path: {model_export_path}")
    model_export_path.unlink()
joblib.dump(stack_model, model_export_path)

### Prelim results ###
train_proba = stack_model.predict_proba(X_train)[:, 1]  # type: ignore
train_score = roc_auc_score(y_train, train_proba)
print(f"Train AUROC: \t{train_score:.3f}")